## RQ1: Predictive Performance & Key Lending Drivers
#### Group 6 | Francesco Biedermann (FB2709) | Brian Hsu (CH4004) | Phoebe Zhao (PYZ2001) 
#### APAN5205 Applied Machine Learning 2 | Prof. Andrew Assing

This notebook addresses Research Question 1: *"To what extent can mortgage approval outcomes be predicted using applicant, loan, and property characteristics available at the time of application, and which factors appear to play the largest role in these decisions?"*

We train and compare three classification models:
- Logistic Regression (with Lasso and Ridge regularization)
- Decision Tree
- Random Forest (using 5-fold cross-validation for hyperparameter tuning)

Demographic variables are excluded from the predictive models and reserved for the fairness analysis in RQ2.

### Step 1 | Setup and Data Loading
We begin by importing the required libraries and loading the cleaned HMDA dataset produced during data preparation. The dataset contains 433,173 mortgage application records with 32 columns and zero missing values.

In [1]:
# Step 1: Setup and Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, ConfusionMatrixDisplay
)

# Load cleaned data
df = pd.read_csv('../data/hmda_2024_cleaned.csv')

print(f"Dataset shape: {df.shape}")
print(f"Approval rate: {df['approved'].mean()*100:.2f}%")
print(f"\nTarget distribution:")
print(f"  Approved (1): {(df['approved']==1).sum():,}")
print(f"  Denied   (0): {(df['approved']==0).sum():,}")

Dataset shape: (433173, 32)
Approval rate: 77.36%

Target distribution:
  Approved (1): 335,087
  Denied   (0): 98,086


### Step 2 | Feature Selection and Preparation
We separate the target variable from the features and apply several preparation steps:

- **Demographic exclusion:** The derived race, ethnicity, and sex columns are excluded from the predictive models, consistent with our research design
- **Constant feature removal:** The "reverse_mortgage" column contains only a single value after data cleaning and provides no discriminative information. Therefore, we dropped it
- **One-hot encoding:** The "state_code" column (54 U.S. states and territories) is one-hot encoded with "drop_first=True" to avoid multicollinearity, yielding 53 binary indicator columns
- All other features are already encoded as numeric values from the data cleaning phase

In [2]:
# Step 2: Feature Selection and Preparation

# Define target
y = df['approved']

# Columns to exclude from modeling
exclude_cols = ['approved', 'derived_race', 'derived_ethnicity', 'derived_sex']

# Drop constant column (reverse_mortgage has only one unique value)
exclude_cols.append('reverse_mortgage')
print(f"Dropping 'reverse_mortgage' — only one unique value: {df['reverse_mortgage'].unique()}")

# Select feature columns
feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].copy()

# One-hot encode state_code (54 categories)
X = pd.get_dummies(X, columns=['state_code'], drop_first=True)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"  - Original numeric/categorical features: {len(feature_cols) - 1}")
print(f"  - State dummy variables: {X.shape[1] - (len(feature_cols) - 1)}")
print(f"\nTarget distribution:")
print(y.value_counts())

Dropping 'reverse_mortgage' — only one unique value: [2]

Feature matrix shape: (433173, 79)
Number of features: 79
  - Original numeric/categorical features: 26
  - State dummy variables: 53

Target distribution:
approved
1    335087
0     98086
Name: count, dtype: int64


### Step 3 | Train-Test Split and Scaling
We split the data 70/30 into training and test sets, stratifying on the target variable to preserve the approval rate in both partitions. Features are then standardized using "StandardScaler", which is fit on the training set and applied to both sets to prevent data leakage.

In [3]:
# Step 3: Train-Test Split and Scaling

# 70/30 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:     {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nApproval rate - Train: {y_train.mean()*100:.2f}%")
print(f"Approval rate - Test:  {y_test.mean()*100:.2f}%")

# Standardize features (fit on train, transform both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Keep column names for interpretability
feature_names = X_train.columns.tolist()

print(f"\nScaling complete. Feature matrix shape: {X_train_scaled.shape}")

Training set: 303,221 rows (70.0%)
Test set:     129,952 rows (30.0%)

Approval rate - Train: 77.36%
Approval rate - Test:  77.36%

Scaling complete. Feature matrix shape: (303221, 79)
